### RAG Evaluation:
using Rouge and Blue

In [3]:
#load the API

import os 
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=True)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
print("API Key Loaded:", os.getenv("GROQ_API_KEY")[:10] if os.getenv("GROQ_API_KEY") else "FAILED TO LOAD")

#Test the API KEY
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI #alternative 

llm_model = ChatGroq(model ="llama-3.3-70b-versatile" )
print(llm_model.invoke("hello, tell me about yourself in few words").content)


API Key Loaded: gsk_8yEZ0T


f:\MLwithpython\LLM_2026\llm_2026_env\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


I'm an AI, here to help and chat.


In [15]:
#Creating a simple RAG
from langchain_community.vectorstores import FAISS

#----------- Load the Embedding model ---------------------
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": False}
embeddings = HuggingFaceEmbeddings(model_name=model_name,model_kwargs=model_kwargs,encode_kwargs=encode_kwargs)

#load the embedding model
vc_db = FAISS.load_local(r"..\vectordb\faiss_index", embeddings,allow_dangerous_deserialization=True)

#Initialize the retrievaer
retriever = vc_db.as_retriever()

from langchain_core.prompts import ChatPromptTemplate
# RAG Prompt
rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the question using only the information provided
in the context.

If the answer is not available in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer:
""")


def rag(question: str):

    # Step 1: Retrieve relevant documents
    documents = retriever.invoke(question)

    # Step 2: Convert documents into context
    context = "\n\n".join(
        doc.page_content
        for doc in documents
    )

    # Step 3: Create the prompt
    prompt = rag_prompt.invoke({
        "context": context,
        "question": question
    })

    # Step 4: Call the LLM
    response = llm_model.invoke(prompt)

    return {"answer": response.content,
           "documents": documents}
        #response.content #this will return str

# Test
question = "What is the main topic of the documents?"
answer = rag(question)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2581.62it/s]


In [18]:
print(answer['documents'])

[Document(id='5daede5f-6407-4345-b756-1ab3b68790da', metadata={'producer': 'Skia/PDF m151', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36', 'creationdate': '2026-08-13T16:08:35+00:00', 'source': '../bootcamp\\sample_docs\\iphone_glass_repair.pdf', 'file_path': '../bootcamp\\sample_docs\\iphone_glass_repair.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': 'iPhone Broken Glass Safety - Apple Support (IN)', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-13T16:08:35+00:00', 'trapped': '', 'modDate': "D:20260813160835+00'00'", 'creationDate': "D:20260813160835+00'00'", 'page': 3}, page_content='7. Remove the sanding cover and vacuum your workspace and the iPhone again. Wipe the back glass with an ethanol\nwipe or IPA wipe.\n8. Ensure that the sanded area appears shiny. If the liquid from the wipe forms into beads or the area doesn’t appear\nshiny, repeat steps 5 through 7.\nImportant: Failur

In [23]:
# ============================================================
# RAG EVALUATION USING BLEU + ROUGE
# ============================================================

import nltk
import pandas as pd

from nltk.translate.bleu_score import (sentence_bleu,SmoothingFunction )

from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score

# ============================================================
# STEP 1: Download NLTK resources
# ============================================================

# Usually not required for BLEU, but safe to initialize NLTK.
#nltk.download("punkt", quiet=True)


# ============================================================
# STEP 2: TEST DATASET
# ============================================================

test_data = [

    {
        "question": "What is the main topic of the documents?",

        "reference": (
            "The main topic of the documents is "
            "financial performance and revenue analysis."
        )
    },

    {
        "question": "What was the company revenue in 2025?",

        "reference": (
            "The company revenue in 2025 was "
            "10 million dollars."
        )
    },

    {
        "question": "Which department is discussed in the report?",

        "reference": (
            "The report discusses the Finance department."
        )
    }

]


# ============================================================
# STEP 3: CREATE ROUGE SCORER
# ============================================================

rouge = rouge_scorer.RougeScorer(
    [
        "rouge1",
        "rouge2",
        "rougeL"
    ],
    use_stemmer=True
)


# ============================================================
# STEP 4: BLEU FUNCTION
# ============================================================

def calculate_bleu(reference, generated):

    # Convert text into tokens

    reference_tokens = (
        reference
        .lower()
        .split()
    )

    generated_tokens = (
        generated
        .lower()
        .split()
    )


    # Calculate BLEU

    score = sentence_bleu(

        [reference_tokens],
        generated_tokens,
        smoothing_function=(
            SmoothingFunction().method1
        )
    )


    return score


# ============================================================
# STEP 5: ROUGE FUNCTION
# ============================================================

def calculate_rouge(
    reference,
    generated
):

    scores = rouge.score(
        reference,
        generated
    )


    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure
    }


# ============================================================
# STEP 5.1: METEOR FUNCTION
# ============================================================
def calculate_meteor(reference, generated):

    reference_tokens = reference.lower().split()
    generated_tokens = generated.lower().split()
    score = meteor_score([reference_tokens],generated_tokens    )
    return score

# ============================================================
# STEP 6: EVALUATE ONE QUESTION
# ============================================================

def evaluate_question( question, reference):

    # --------------------------------------------------------
    # Call your RAG
    # --------------------------------------------------------

    result = rag(question)
    #check what RAG actually returns
    print("Result Type-------------------", type(result))


    # --------------------------------------------------------
    # Get generated answer
    # --------------------------------------------------------

    generated = result[ "answer" ]

    # --------------------------------------------------------
    # BLEU
    # --------------------------------------------------------

    bleu = calculate_bleu(reference, generated )

    # --------------------------------------------------------
    # ROUGE
    # --------------------------------------------------------

    rouge_scores = calculate_rouge(reference, generated )


    # --------------------------------------------------------
    # ROUGE
    # --------------------------------------------------------
    meteor = calculate_meteor(reference, generated)

    # --------------------------------------------------------
    # Return results
    # --------------------------------------------------------

    return {
        "question": question,
        "reference": reference,
        "generated": generated,
        "bleu": bleu,
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"],
        "meteor": meteor
    }


# ============================================================
# STEP 7: RUN EVALUATION
# ============================================================

results = []

for item in test_data:

    print("\nEvaluating:" )
    print(item["question"])

    result = evaluate_question(
        item["question"],
        item["reference"])

    results.append(result)


# ============================================================
# STEP 8: CREATE DATAFRAME
# ============================================================

results_df = pd.DataFrame(results)

# ============================================================
# STEP 9: DISPLAY RESULTS
# ============================================================

print("\n")
print("================================================")
print("RAG EVALUATION RESULTS")
print( "================================================")

print(
    results_df[
        [
            "question",
            "bleu",
            "rouge1",
            "rouge2",
            "rougeL",
            "meteor"
        ]
    ]
)


# ============================================================
# STEP 10: AVERAGE SCORES
# ============================================================

average_bleu = results_df["bleu"].mean()
average_rouge1 = results_df["rouge1"].mean()
average_rouge2 = results_df["rouge2"].mean()
average_rougeL = results_df["rougeL"].mean()
average_meteor = results_df["meteor"].mean()


print("\n")
print("================================================")
print( "AVERAGE RAG SCORES")
print("================================================")
print(f"Average BLEU   : {average_bleu:.4f}")
print(f"Average ROUGE-1: {average_rouge1:.4f}")
print(f"Average ROUGE-2: {average_rouge2:.4f}")
print(f"Average ROUGE-L: {average_rougeL:.4f}")
print(f"Average METEOR: {average_meteor:.4f}")


Evaluating:
What is the main topic of the documents?
Result Type------------------- <class 'dict'>

Evaluating:
What was the company revenue in 2025?
Result Type------------------- <class 'dict'>

Evaluating:
Which department is discussed in the report?
Result Type------------------- <class 'dict'>


RAG EVALUATION RESULTS
                                       question      bleu    rouge1    rouge2  \
0      What is the main topic of the documents?  0.148527  0.318182  0.238095   
1         What was the company revenue in 2025?  0.024512  0.111111  0.000000   
2  Which department is discussed in the report?  0.027776  0.133333  0.000000   

     rougeL    meteor  
0  0.318182  0.342566  
1  0.111111  0.056180  
2  0.133333  0.080645  


AVERAGE RAG SCORES
Average BLEU   : 0.0669
Average ROUGE-1: 0.1875
Average ROUGE-2: 0.0794
Average ROUGE-L: 0.1875
Average METEOR: 0.1598


For example:

Reference: Revenue increased by 20% in 2025.
Generated:The company's sales grew by 20% during 2025.

This may be a very good answer, but exact word-overlap metrics can penalize it.
That's why for your RAG project recommend this evaluation stack

              RAG Evaluation
                    │
       ┌────────────┴────────────┐
       │                         │
 Answer Quality             Retrieval Quality
       │                         │
 ┌─────┼─────┐             ┌─────┼──────┐
 ▼     ▼     ▼             ▼     ▼      ▼
BLEU ROUGE METEOR        Recall Precision MRR
       │
       ▼
 BERTScore
       │
       ▼
 Grounding / Faithfulness

*Retrieval quality asks: "Did we retrieve the right information?"
*Answer quality asks: "Did the LLM use that information to produce a good answer?"

In [26]:
# ============================================================
# RAG RETRIEVAL QUALITY EVALUATION
# ============================================================
#
# Metrics:
#   1. Precision@K
#   2. Recall@K
#   3. Hit Rate@K:Measures whether at least one relevant document appears in the top K retrieved documents. 1 - relevant doc found else 0
#   4. MRR (Mean Reciprocal Rank):Measures how high the first relevant document appears in the ranked results. Higher MRR means relevant documents appear closer to the top.
#
# This code evaluates ONLY the RETRIEVAL component.
# It does NOT evaluate the LLM-generated answer.
#
# Assumption:
#   Your existing FAISS vector database is called: vc_db
#
#   And each Document contains metadata such as: doc.metadata["source"]
#
# ============================================================


import pandas as pd

# ============================================================
# STEP 1: TEST DATASET
# ============================================================

test_data = [

    {
        "question": "What is the main topic of the documents?",
        "relevant_sources": ["finance_2025.pdf" ] #hese must match the metadata in your actual FAISS documents.
    },


    {
        "question":"What was the company revenue in 2025?",
        "relevant_sources": ["finance_2025.pdf"]
    },


    {
        "question":"Which department is discussed in the report?",
        "relevant_sources": [ "finance_2025.pdf"]
    }
]


# ============================================================
# STEP 2: RETRIEVE DOCUMENTS
# ============================================================


def retrieve_documents(question, k=5):
    retriever = vc_db.as_retriever(search_kwargs={"k": k })
    documents = retriever.invoke(question)
    return documents

# ============================================================
# STEP 3: GET DOCUMENT SOURCE
# ============================================================

def get_source(doc):
    return doc.metadata.get("source", "" )

# ============================================================
# STEP 4: PRECISION@K
# ============================================================
#
# Precision@K answers:
#
# "Of the K documents retrieved,
#  how many are relevant?"
#
# Example:
# Retrieved 5
# Relevant 3
#
# Precision@5 = 3 / 5 = 0.60
# ============================================================


def calculate_precision_at_k(retrieved_documents,relevant_sources,k):

    retrieved_documents = (retrieved_documents[:k] )
    relevant_count = 0

    for doc in retrieved_documents:
        source = get_source(doc)
        if source in relevant_sources:
            relevant_count += 1

    if k == 0:
        return 0.0

    precision = (relevant_count / k)

    return precision


# ============================================================
# STEP 5: RECALL@K
# ============================================================
#
# Recall@K answers:
# "Of all the relevant documents,
#  how many did we retrieve?"
# ============================================================


def calculate_recall_at_k(retrieved_documents,relevant_sources,k):
    retrieved_documents = ( retrieved_documents[:k])

    retrieved_sources = set(get_source(doc)
        for doc in retrieved_documents )

    relevant_sources = set(relevant_sources)

    if len(relevant_sources) == 0:
        return 0.0

    found_documents = (retrieved_sources & relevant_sources)
    recall = ( len(found_documents) /len(relevant_sources))

    return recall


# ============================================================
# STEP 6: HIT RATE@K
# ============================================================
#
# Hit Rate@K answers:
#
# "Did we retrieve at least one relevant
#  document in the top K?"
#
# Result:
# 1.0 = Hit
# 0.0 = Miss
# ============================================================


def calculate_hit_rate_at_k(retrieved_documents,relevant_sources, k):

    retrieved_documents = (retrieved_documents[:k])
    retrieved_sources = set(get_source(doc)for doc in retrieved_documents)

    for source in relevant_sources:
        if source in retrieved_sources:
            return 1.0

    return 0.0


# ============================================================
# STEP 7: MEAN RECIPROCAL RANK (MRR)
# ============================================================
#
# Reciprocal Rank measures where the FIRST relevant
# document appears.
#
# Position 1 → 1 / 1 = 1.00
# Position 2 → 1 / 2 = 0.50
# Position 3 → 1 / 3 = 0.33
# Position 4 → 1 / 4 = 0.25
# Position 5 → 1 / 5 = 0.20
# ============================================================


def calculate_reciprocal_rank(retrieved_documents,relevant_sources):
    relevant_sources = set(relevant_sources)

    for rank, doc in enumerate(retrieved_documents, start=1):
        source = get_source(doc)

        if source in relevant_sources:
            return 1.0 / rank
    return 0.0


# ============================================================
# STEP 8: EVALUATE ONE QUESTION
# ============================================================


def evaluate_retrieval(question,relevant_sources,k=5):

    # --------------------------------------------------------
    # Retrieve documents
    # --------------------------------------------------------

    documents = retrieve_documents(question,k)

    # --------------------------------------------------------
    # Precision@K
    # --------------------------------------------------------

    precision = calculate_precision_at_k(documents,relevant_sources,k)

    # --------------------------------------------------------
    # Recall@K
    # --------------------------------------------------------

    recall = calculate_recall_at_k(documents,relevant_sources,k)

    # --------------------------------------------------------
    # Hit Rate@K
    # --------------------------------------------------------

    hit_rate = calculate_hit_rate_at_k(documents,relevant_sources,k)

    # --------------------------------------------------------
    # Reciprocal Rank
    # --------------------------------------------------------

    reciprocal_rank = (calculate_reciprocal_rank(documents,relevant_sources ))

    # --------------------------------------------------------
    # Return evaluation result
    # --------------------------------------------------------

    return {
        "question":
            question,

        "precision":
            precision,

        "recall":
            recall,

        "hit_rate":
            hit_rate,

        "mrr":
            reciprocal_rank,

        "documents":
            documents
    }


# ============================================================
# STEP 9: RUN RETRIEVAL EVALUATION
# ============================================================

results = []

for item in test_data:

    print("\nEvaluating:")
    print(item["question"])

    result = evaluate_retrieval(question =item["question"],
        relevant_sources =item["relevant_sources"],k = 5 )

    results.append(result)

# ============================================================
# STEP 10: CREATE DATAFRAME
# ============================================================
#
# We deliberately don't put "documents" into the DataFrame
# because the Document objects can create very large output.
# ============================================================


results_df = pd.DataFrame([{"question":result["question"],
            "precision@5": result["precision"],
            "recall@5":result["recall"],
            "hit_rate@5":result["hit_rate"],
            "mrr":result["mrr"]}
        for result in results
    ])


# ============================================================
# STEP 11: DISPLAY RESULTS
# ============================================================

print("\n")
print("================================================")
print("RAG RETRIEVAL QUALITY RESULTS")
print("================================================")

print(results_df.to_string(index=False ))

# ============================================================
# STEP 12: AVERAGE SCORES
# ============================================================

average_precision = (results_df["precision@5"].mean())
average_recall = (results_df["recall@5" ].mean())
average_hit_rate = (results_df["hit_rate@5"].mean())
average_mrr = (results_df["mrr"].mean())

# ============================================================
# STEP 13: DISPLAY AVERAGE SCORES
# ============================================================

print("\n")
print("================================================")
print("AVERAGE RETRIEVAL QUALITY")
print("================================================")
print(f"Average Precision@5:" f"{average_precision:.4f}")
print(f"Average Recall@5:"f"{average_recall:.4f}")
print( f"Average Hit Rate@5:"f"{average_hit_rate:.4f}")
print(f"Average MRR:" f"{average_mrr:.4f}")


# ============================================================
# STEP 14: OPTIONAL - SHOW RETRIEVED SOURCES
# ============================================================
#
# Use this section when you want to inspect what FAISS
# actually retrieved.
# ============================================================

print("\n")
print("================================================")
print( "RETRIEVED DOCUMENTS")
print( "================================================")

for result in results:

    print("\nQuestion:" )
    print(result["question"] )
    print("\nRetrieved Sources:" )


    for rank, doc in enumerate(result["documents"], start=1 ):

        print( f"{rank}. "
            f"{get_source(doc)}" )


Evaluating:
What is the main topic of the documents?

Evaluating:
What was the company revenue in 2025?

Evaluating:
Which department is discussed in the report?


RAG RETRIEVAL QUALITY RESULTS
                                    question  precision@5  recall@5  hit_rate@5  mrr
    What is the main topic of the documents?          0.0       0.0         0.0  0.0
       What was the company revenue in 2025?          0.0       0.0         0.0  0.0
Which department is discussed in the report?          0.0       0.0         0.0  0.0


AVERAGE RETRIEVAL QUALITY
Average Precision@5:0.0000
Average Recall@5:0.0000
Average Hit Rate@5:0.0000
Average MRR:0.0000


RETRIEVED DOCUMENTS

Question:
What is the main topic of the documents?

Retrieved Sources:
1. ../bootcamp\sample_docs\iphone_glass_repair.pdf
2. ../bootcamp\sample_docs\iphone_glass_repair.pdf
3. ../bootcamp\sample_docs\iphone_glass_repair.pdf
4. ../bootcamp\sample_docs\iphone_glass_repair.pdf
5. ../bootcamp\sample_docs\iphone_glass_rep